# See-through on Google Colab

Single image -> layered PSD (up to 23 layers) with the See-through pipeline.

**Runtime:** `Runtime > Change runtime type > GPU`. L4 or A100 recommended (bf16 support).
T4 (free tier) has no native bf16 and only ~12.7GB system RAM, so it may be very slow or crash.

Run the cells top to bottom. Scripts run in a subprocess (`!python`), so no runtime restart is needed after installing.

In [ ]:
# 1. Check GPU
import torch, subprocess
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > GPU'
name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
major, _ = torch.cuda.get_device_capability(0)
print(f'{name}, {vram_gb:.1f} GB VRAM, compute capability {major}.x')
if major < 8:
    print('WARNING: this GPU has no native bf16 (needs Ampere or newer, e.g. L4/A100). Expect slow runs or errors.')
!free -g | head -2

## 2. (Optional) Cache models on Google Drive

Models (~15GB) are downloaded from HuggingFace every new session (~10 min).
Set `USE_DRIVE_CACHE = True` to keep them on Drive and skip the download next time.

In [ ]:
USE_DRIVE_CACHE = False  #@param {type:"boolean"}
import os
if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'
    os.makedirs(os.environ['HF_HOME'], exist_ok=True)
    print('HF_HOME =', os.environ['HF_HOME'])

In [ ]:
# 3. Clone and install
REPO_URL = 'https://github.com/xoxwgys56/see-through.git'  #@param {type:"string"}
%cd /content
![ -d see-through ] || git clone --depth 1 {REPO_URL} see-through
%cd /content/see-through
![ -e assets ] || ln -s common/assets assets
# Colab already ships CUDA torch; skip Qt UI and ipykernel pins so the Colab kernel is untouched
!grep -vE '^(PyQt6|qtpy|ipykernel)' requirements.txt > req-colab.txt
!pip install -q -r req-colab.txt

In [ ]:
# 4. Input image: upload your own, or leave unchecked to use the bundled test image
UPLOAD = True  #@param {type:"boolean"}
import os
os.makedirs('/content/inputs', exist_ok=True)
if UPLOAD:
    from google.colab import files
    for fn, data in files.upload().items():
        open(os.path.join('/content/inputs', fn), 'wb').write(data)
else:
    !cp assets/test_image.png /content/inputs/
!ls /content/inputs

In [ ]:
# 5. Run (all images in /content/inputs)
RESOLUTION = 1280  #@param [768, 1024, 1280] {type:"raw"}
GROUP_OFFLOAD = vram_gb < 20  # README: ~10GB peak at 1280 with group offload
TBLR_SPLIT = False  #@param {type:"boolean"}
import time
flags = ' --group_offload' if GROUP_OFFLOAD else ''
flags += ' --tblr_split' if TBLR_SPLIT else ''
print('group_offload =', GROUP_OFFLOAD)
t0 = time.time()
!python inference/scripts/inference_psd.py --srcp /content/inputs --save_dir /content/outputs --save_to_psd --resolution {RESOLUTION}{flags}
print(f'elapsed: {(time.time() - t0) / 60:.1f} min')

In [ ]:
# 6. Preview and download
import glob
from psd_tools import PSDImage
from IPython.display import display
for p in sorted(glob.glob('/content/outputs/*.psd')):
    if p.endswith('_depth.psd'):
        continue
    psd = PSDImage.open(p)
    print(p, psd.size, [l.name for l in psd.descendants()])
    im = psd.composite()
    im.thumbnail((512, 512))
    display(im)

!cd /content && zip -qr see-through-outputs.zip outputs
from google.colab import files
files.download('/content/see-through-outputs.zip')